## 1. Introdução e Configuração do Ambiente Estendido
Este notebook documenta o processo experimental e reprodutível de treino de um modelo de Visão por Computador baseado na arquitetura YOLOv8 (especificamente a variante `yolov8s.pt`), focado na tarefa industrial de deteção de luvas de proteção em ambiente laboratorial na industria farmaceutica.

O pipeline foi desenhado para correr no ecossistema Google Colab com aceleração gráfica por hardware (GPU). A primeira fase consiste em estabelecer uma ligação persistente com o Google Drive para permitir:
1. A importação segura de scripts utilitários customizados (como o módulo de aumentos estatísticos).
2. O armazenamento centralizado e persistente da chave privada de API do Roboflow (evitando a exposição pública de credenciais).
3. A gravação automatizada dos *checkpoints* do modelo (`best.pt` e `last.pt`) e dos relatórios gráficos gerados a cada época, prevenindo a perda de progresso devido a quebras de sessão ou tempos limite do ambiente Colab.

In [ ]:
# 1. Configuração do Ambiente e Importação do Google Drive
from google.colab import drive
import shutil
import os

# Montar o sistema de ficheiros do Google Drive no caminho local do ambiente temporário (/content/drive)
drive.mount('/content/drive')

# Definição do caminho base do projeto onde os ficheiros persistentes estão armazenados no Drive
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/gloves_project"

# Copiar o script de suporte de data augmentation customizado para a raiz do ambiente local de execuçãoshutil.copy(f"{PROJECT_DIR}/augmentation.py", "/content/augmentation.py")
shutil.copy(f"{PROJECT_DIR}/augmentation.py", "/content/augmentation.py")

# Copiar o ficheiro de texto que contém o token privado de autenticação da API do Roboflow
shutil.copy(f"{PROJECT_DIR}/api_key", "/content/api_key")

## 2. Ecossistema de Dependências Técnicas e Ativação do TensorBoard

Para além da biblioteca nativa da Ultralytics, o projeto integra o ecossistema **Albumentations**.

O **Albumentations** é uma biblioteca de código aberto focada no processamento de imagens e na aplicação de técnicas avançadas de *Data Augmentation* (aumento artificial de dados) para Aprendizagem Profunda (*Deep Learning*). 

Adicionalmente, ativamos o suporte ao **TensorBoard** através da CLI do YOLO. Isto configura um servidor local de telemetria onde podemos acompanhar visualmente as curvas de perda (*Box Loss*, *Class Loss*, *DFL Loss*) e métricas de validação por época, servindo de ferramenta de diagnóstico contra o sobreajustamento (*overfitting*).

In [ ]:
# Instalação silenciosa (-q) das dependências estruturais do projeto
!pip install ultralytics roboflow albumentations opencv-python pyyaml -

# Configurar as definições globais da CLI do YOLO para forçar a integração automática com o TensorBoard
!yolo settings tensorboard=True

from IPython import display
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import albumentations
import roboflow
import ultralytics
from ultralytics import YOLO

# Limpar todo o histórico de logs de instalação textuais do ecrã para melhor legibilidade
display.clear_output()

# Executar o diagnóstico de integridade da Ultralytics para validar a GPU ativa e os recursos do sistema
ultralytics.checks()


## 3. Aquisição de Dados via Roboflow API

O conjunto de dados original foi previamente anotado, revisto e estruturado na plataforma Roboflow. Através do SDK oficial, efetuamos a autenticação automática em modo não interativo utilizando a chave copiada do Drive. O download descarrega o projeto em formato YOLOv8, o qual cria nativamente as diretorias separadas necessárias para o treino robusto: `train`, `valid` e `test`, acompanhadas pelo ficheiro descritivo de metadados `data.yaml`.

In [ ]:
# Inicializar o processo de login não interativo no ecossistema Roboflow
roboflow.login()

# Ler a chave de API de forma isolada, limpando quebras de linha com o método .strip()
with open('api_key', "r") as file:
    api_key = file.read().strip()

# Instanciar a ligação com a plataforma injetando a credencial obtida
rf = roboflow.Roboflow(api_key=api_key)

# Mapear o Workspace académico do projeto e aceder ao ID específico do repositório do trabalho
project = rf.workspace("ruis-workspace-qvcoy").project("trabalho_ia-opgfv")

# Descarregar a versão 2 estável do dataset convertida especificamente para a estrutura YOLOv8
dataset = project.version(2).download("yolov8")



## 4. Execução da Pipeline de Data Augmentation Customizada

A fim de mitigar problemas de desbalanceamento estatístico entre as classes, invocamos o script local `augmentation.py` em modo balanceado (`--mode balanced`). O algoritmo analisa a distribuição populacional das classes e aplica transformações matemáticas nas imagens sub-representadas até atingir o teto fixado de 2000 instâncias, aumentando drasticamente a capacidade de generalização do modelo final.

In [ ]:
import subprocess
from pathlib import Path

# Definir a localização local onde as pastas do dataset foram extraídas pelo Roboflow
DATASET_LOCATION = "/content/Trabalho_IA-2"

# Criar um marcador (ficheiro oculto) para garantir a idempotência do processo (não repetir aumentos desnecessariamente)
done_flag = Path(DATASET_LOCATION) / "train" / ".augmented_done"

# Executar a pipeline de data augmentation caso ainda não tenha sido processada
f done_flag.exists():
    print("Processamento de Data Augmentation ignorado: Já aplicado com sucesso anteriormente.")
    print(done_flag.read_text())
else:
    print("A iniciar o pipeline de Data Augmentation customizado no split de treino...")
    # Executar o script via subprocesso passando os parâmetros de balanceamento e meta de imagens
    result = subprocess.run(
        ["python", "augmentation.py", "--data", DATASET_LOCATION, "--mode", "balanced", "--target", "2000"],
        check=True
    )

## 5. Configuração Fina de Hiperparâmetros e Execução do Treino

Esta é a fase crítica do projeto. Instanciamos a rede carregando os pesos base pré-treinados do modelo `yolov8s.pt` (YOLOv8 Small). Esta escolha provou ser o balanço ideal entre custo computacional (velocidade de inferência) e capacidade de extração de características em redes convolucionais.

### Justificação Científica da Escolha dos Hiperparâmetros Vencedores:
A seleção e o ajuste fino dos hiperparâmetros foram fundamentais para mitigar o enviesamento (*bias*) e garantir a convergência estocástica do modelo. A parametrização final adotada reflete a combinação empírica mais robusta para o nosso domínio de aplicação:

* **`epochs=100`**: O estabelecimento do teto macro em 100 épocas garantiu uma janela temporal computacional suficiente para superar a fase inicial de subajustamento (*underfitting*).
* **`lr0=0.001`**: A taxa de aprendizagem inicial (*Initial Learning Rate*) fixada em 0.001 revelou-se o equilíbrio ideal entre estabilidade e velocidade de convergência.
* **`patience=20`**: A parametrização do critério de paragem antecipada (*Early Stopping*) para 20 épocas atuou como o principal mecanismo de regularização dinâmica contra o sobreajustamento (*overfitting*). 
* **`optimizer='SGD'`**: A especificação explícita do otimizador para **SGD** (*Stochastic Gradient Descent*) define o modelo matemático de retropropagação e atualização de pesos baseado na abordagem clássica de gradiente estocástico.


In [ ]:
from ultralytics.utils.callbacks.base import on_fit_epoch_end
import pandas as pd
from pathlib import Path
from IPython.display import clear_output, display

# Inicializar o modelo carregando a arquitetura YOLOv8 Small com pesos pré-treinados no COCO dataset
model = YOLO("yolov8s.pt")

# Definição dos caminhos absolutos de origem (SSD local) e destino (Google Drive via Rede)
RESULTS_DIR = "/content/runs"
DRIVE_DIR   = "/content/drive/MyDrive/Colab Notebooks/gloves_project/runs"
EXPERIMENT  = "yolov8s"

def on_train_end(trainer):
    """
    Gatilho de finalização do treino.
    Realiza a transferência em massa (I/O) dos resultados do ambiente temporário
    para o armazenamento em nuvem não volátil.
    """

    # Mapear as diretorias locais e remotas usando a biblioteca orientada a objetos Path
    src = Path(RESULTS_DIR) / EXPERIMENT
    dst = Path(DRIVE_DIR) / EXPERIMENT

    # Executar a cópia recursiva da árvore de diretórios.
    # O parâmetro dirs_exist_ok=True previne exceções caso a pasta de destino já tenha sido criada
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"[callback] Training complete — full results copied to Drive.")


# Injetar os callbacks customizados no ciclo de vida do modelo antes de iniciar o método fit()
model.add_callback("on_train_end", on_train_end)

# Bloco de execução do treino com a parametrização explícita do dicionário da Ultralytics
results = model.train(
    # ── Ficheiros de Configuração e Dados ──────────────────────────────────────
    data = f"{DATASET_LOCATION}/data.yaml",

    # ── Hiperparâmetros de Otimização e Aprendizagem  ──────────────────────────-
    epochs = 100, # Número máximo de passagens completas pelo dataset de treino
    patience=20,            # Paragem antecipada se a perda de validação estagnar durante 20 épocas consecutivas
    batch=16,               # Número de imagens processadas simultaneamente antes de atualizar os gradientes da rede
    imgsz=640,              # Resolução de redimensionamento das imagens de entrada (padrão de alta fidelidade YOLO)
    lr0=0.001,               # Taxa de aprendizagem inicial para os algoritmos de descida de gradiente
    lrf=0.01,               # Taxa de aprendizagem final 
    momentum=0.937,         # Fator de inércia para acelerar os vetores de gradiente na direção certa e amortecer oscilações
    weight_decay=0.0005,    # Penalização de regularização L2 aplicada aos pesos para prevenir Overfitting nas camadas densas
    warmup_epochs=3.0,      # Épocas iniciais com lr reduzido para estabilizar os gradientes antes do treino principal
    warmup_momentum=0.8,    # Momentum inicial durante a fase de aquecimento
    warmup_bias_lr=0.1,     # Taxa de aprendizagem inicial aplicada especificamente aos vieses (bias) no aquecimento
    box=7.5,                # Peso/importância da perda da caixa delimitadora (GIoU Loss) na função de erro total
    cls=0.5,                # Peso atribuído à perda de classificação (BCELoss) no cálculo de erro do modelo
    dfl=1.5,

    # ── Configurações do Otimizador e Dispositivo de Hardware ──────────────────
    optimizer='SGD',        # Seleção otimizador SGD do algoritmo
    device=0,               # Index da GPU atribuída pelo Google Colab (0 força o uso do hardware NVIDIA acelerado)
    workers=8,              # Número de threads paralelas na CPU dedicadas exclusivamente ao carregamento e preparação de dados

    # ── Modos de Regularização e Augmentations em Tempo de Treino ──────────────
    hsv_h=0.015,            # Fração de alteração máxima aleatória da tonalidade da imagem (Hue)
    hsv_s=0.7,              # Fração de alteração máxima aleatória da saturação da imagem (Saturation)
    hsv_v=0.4,              # Fração de alteração máxima aleatória do brilho/valor da imagem (Value)
    degrees=0.0,            # Grau de rotação aleatória desativado (já coberto pelo nosso script Albumentations)
    translate=0.1,          # Fator de translação horizontal e vertical aleatória das imagens (10% do tamanho)
    scale=0.5,              # Fator de escala/zoom aleatório (permite simular objetos a várias distâncias da câmara)
    shear=0.0,              # Ângulo de cisalhamento desativado
    perspective=0.0,        # Distorção de perspetiva 3D desativada
    flipud=0.0,             # Inversão vertical desativada (as luvas raramente aparecem invertidas de pernas para o ar)
    fliplr=0.0,             # Inversão horizontal desativada (simula luva esquerda vs luva direita)
    mosaic=0.5,             # Probabilidade de combinar 4 imagens numa só durante o treino (aumento de contexto espacial)
    mixup=0.0,              # Mistura de duas imagens diferentes desativada (pode criar ruído excessivo em deteções pequenas)
    copy_paste=0.0,         # Técnica de copiar objetos de uma imagem e colar noutra desativada

    # ── Destino, Logs e Outputs Persistentes ──────────────────────────────────
    project=DRIVE_RUNS_DIR, # Grava diretamente para a tua pasta mapeada do Google Drive
    name="yolov8s",         # Nome específico da pasta que conterá os pesos guardados e gráficos de validação
    exist_ok=False,         # Se a pasta já existir, lança um erro em vez de sobreescrever dados antigos acidentalmente
    save=True,              # Ativa a gravação física dos ficheiros de pesos (.pt) e checkpoints ao longo do processo
    save_period=-1,         # Desativa a gravação por intervalos fixos de épocas (apenas grava o melhor e o último por defeito)
    pretrained=True,        # Confirma a utilização de pesos pré-treinados ImageNet/COCO em vez de inicialização aleatória
    freeze=0,               # Não congela camadas; permite que todas as camadas sofram fine-tuning para o nosso problema
    verbose=True,           # Imprime no ecrã de forma detalhada as métricas obtidas no final de cada época concluída
    seed=0,                 # Fixa a semente aleatória global garantindo a reprodutibilidade exata dos resultados experimentais
    deterministic=True,     # Obriga a utilização de algoritmos determinísticos na GPU (reprodutibilidade matemática rigorosa)
    val=True,               # Força a execução da validação completa no final de cada época para monitorizar o mAP
    plots=True,             # Cria e guarda automaticamente ficheiros de imagem com gráficos de perdas, matriz de confusão e curvas PR

    # ── Parâmetros Técnicos Avançados de Compilação ───────────────────────────
    resume=False,           # Define se deve continuar um treino interrompido (deve ser False ao iniciar um treino novo)
    amp=True,               # Ativa a Precisão Mista Automática (FP16); acelera o treino e poupa VRAM sem perder precisão
    fraction=1.0,           # Utiliza 100% das imagens disponíveis no conjunto de dados de treino
    profile=False           # Desativa o profiling de velocidade de exportação para ONNX/TensorRT durante a fase de treino
)
    


# 6. Módulo de Inferência Local e Operacionalização do Modelo (Deployment)

Esta secção implementa o script definitivo de inferência e produção para o modelo preditivo (`model.predict()`). Enquanto os módulos anteriores estavam focados no treino e na auditoria estatística comparativa contra subconjuntos anotados (*Ground Truth*), este algoritmo foi desenhado para funcionar de forma autónoma em ambiente de produção ou *deployment*.

O pipeline automatiza o processamento de imagens brutas em tempo real através dos seguintes passos de engenharia:
1. **Instanciação do Ponto de Viragem Ótimo:** Carrega a arquitetura de pesos com o melhor desempenho registado no Drive (neste caso, o checkpoint `yolov8s`).
2. **Filtragem de Extensões Suportadas:** Varre a diretoria local de entrada (`input`) recorrendo a um mapeamento de conjuntos (*set hashing*) para isolar ficheiros de imagem válidos.
3. **Passagem Forward e Controlo de Hiperparâmetros de Inferência:** Executa a inferência na GPU parametrizando limites estritos de confiança (`conf`), supressão de não-máximos baseada em IoU (`iou`) e resolução de tensores (`imgsz`).
4. **Estruturação de Metadados Bidimensionais (JSON):** Desconstrói os tensores de saída da rede para extrair as coordenadas geométricas exatas ($x_1, y_1, x_2, y_2$), a área espacial (*width* e *height*) e os scores de confiança, serializando-os numa estrutura interoperável JSON.
5. **Renderização Vetorial e Escrita em Disco:** Sobrepõe os elementos gráficos à matriz de pixéis original e escreve os artefactos visuais finais na diretoria de saída (`output`).

In [ ]:
import json
import os
from pathlib import Path
import cv2
from ultralytics import YOLO

# ── 1. Configuração de Infraestrutura e Diretórias de Produção ────────────────
# Definição do caminho do modelo selecionado como vencedor após a fase de tuning
MODEL_PATH   = "drive/MyDrive/Colab Notebooks/gloves_project/runs/yolov8s_fase2/weights/best.pt"
INPUT_DIR    = Path("input")
OUTPUT_DIR   = Path("output")

# Garantir a criação das diretorias operacionais no sistema de ficheiros local
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.makedirs("input", exist_ok=True) 

# Conjunto de extensões gráficas suportadas pelo descodificador OpenCV
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

# ── 2. Carregamento da Rede Neuronal Convolucional ───────────────────────────
print(f"[Inference] A inicializar o modelo preditivo a partir de: {MODEL_PATH}")
model = YOLO(MODEL_PATH)

# Mapear de forma iterativa todas as imagens elegíveis presentes na pasta input
image_paths = [
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in SUPPORTED_EXTENSIONS
]

if not image_paths:
    print(f"[AVISO] Nenhuma imagem localizada em '{INPUT_DIR}'. Formatos elegíveis: {SUPPORTED_EXTENSIONS}")

# ── 3. Pipeline de Processamento e Inferência Sequencial ──────────────────────
for image_path in image_paths:
    print(f"A processar frame operacional: {image_path.name}")

    # Executar a previsão (Passagem Forward na rede com os hiperparâmetros operacionais)
    results = model.predict(
        source=str(image_path),
        conf=0.4,           # Limiar mínimo de confiança. Ativações abaixo de 40% são descartadas.
                            # Range: 0.0–1.0. Valores mais baixos aumentam o ruído (Falsos Positivos).
        
        iou=0.7,            # Limiar IoU para o algoritmo de Non-Maximum Suppression (NMS).
                            # Caixas sobrepostas acima de 70% são fundidas (deduplicadas).
                            # Valores mais baixos tornam a eliminação de caixas duplicadas mais estrita.
        
        imgsz=640,          # Redimensionamento do tensor de entrada. Deve coincidir com a escala do treino.
        max_det=300,        # Teto macro de deteções máximas permitidas por imagem.
        device=0,           # Dispositivo de hardware de inferência (0 força a aceleração por GPU NVIDIA).
        verbose=False,      # Ocultar o log padrão textual por imagem na consola.
    )

    result = results[0]  # Extrair o mapa de resultados da imagem corrente

    # ── 4. Desconstrução de Tensores e Extração de Primitivas Geométricas ──────
    detections = []
    for box in result.boxes:
        # Extrair as coordenadas dos cantos da caixa delimitadora em pixéis nativos
        x1, y1, x2, y2 = box.xyxy[0].tolist()   
            
        # Injetar os dados normalizados no dicionário estruturado de metadados
        detections.append({
            "class_id":   int(box.cls),                    # Índice numérico identificador da classe
            "class_name": model.names[int(box.cls)],       # Label textual correspondente à classe mapeada
            "confidence": round(float(box.conf), 4),       # Score probabilístico de confiança arredondado
            "bbox": {
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "x2": round(x2, 2),
                "y2": round(y2, 2),
                "width":  round(x2 - x1, 2),               # Largura computada do objeto na imagem
                "height": round(y2 - y1, 2),              # Altura computada do objeto na imagem
            }
        })

    # ── 5. Serialização de Metadados e Persistência de Resultados ─────────────
    # Escrever o ficheiro de metadados JSON correspondente à imagem

    json_path = OUTPUT_DIR / f"{image_path.stem}.json"
    with open(json_path, "w") as f:
        json.dump(detections, f, indent=2)

    # Invocar a primitiva gráfica .plot() para gerar a matriz de pixéis sobreposta (BGR Array)
    annotated_image = result.plot(
            conf=True,      # Renderizar a percentagem de confiança em cima da caixa
            labels=True,    # Renderizar o nome descritivo da classe
            boxes=True,     # Desenhar o contorno retangular geométrico
            line_width=2,   # Definir a espessura da linha em pixéis
    )

    # Gravar a imagem anotada final no disco rígido através do OpenCV
    output_image_path = OUTPUT_DIR / image_path.name
    cv2.imwrite(str(output_image_path), annotated_image)

    print(f"  → {len(detections)} detection(s) | image saved to '{output_image_path}' | JSON saved to '{json_path}'")

print("\n[Inference] Processamento em lote concluído com sucesso.")

## 7. Exportação de Evidências e Descarregamento dos Resultados

Dado que o ambiente do Google Colab opera sobre instâncias virtuais assíncronas e voláteis, todos os ficheiros armazenados temporariamente na diretoria local `/content/` são destruídos permanentemente assim que a sessão é encerrada ou por inatividade do utilizador.

O bloco final abaixo utiliza os métodos de compressão da biblioteca `shutil` para consolidar as matrizes de pixéis geradas e os respetivos metadados JSON num arquivo unificado. Posteriormente, invoca a API de I/O nativa do ecossistema Colab para mediar a transferência assíncrona do arquivo `.zip` de forma explícita e segura para o armazenamento não-volátil da máquina local do utilizador.

In [ ]:
import shutil
from google.colab import files

print("[Export] A empacotar a diretoria de outputs para arquivo ZIP...")
# Compactar a pasta "output" local gerando o ficheiro resultados_output.zip na raiz do sistema
shutil.make_archive("/content/resultados_output", "zip", "/content/output")

print("[Download] A iniciar transferência assíncrona para o sistema local...")
# Executar o descarregamento apontando de forma estrita para o caminho absoluto do arquivo
files.download("/content/resultados_output.zip")